In [ ]:
# @cr:code_system name='framework_path' id=a3d4551b
# Prepend the Repos checkout of customer_retention to sys.path so the
# live source tree wins over any wheel installed on the cluster (avoids
# stale-symbol ImportErrors during framework development).
#
# Two ways to set FRAMEWORK_REPO_ROOT:
#   1. Set ``CR_FRAMEWORK_REPO_ROOT`` env var in the cluster config, OR
#   2. Edit the literal below to your workspace path, e.g.
#      ``/Workspace/Repos/<user@org.com>/customer_retention``
import os
import sys

FRAMEWORK_REPO_ROOT = os.environ.get(
    "CR_FRAMEWORK_REPO_ROOT",
    "/Workspace/Repos/<USER>/customer_retention",  # TODO: set for your workspace
)
_src = f"{FRAMEWORK_REPO_ROOT}/src"
if _src not in sys.path:
    sys.path.insert(0, _src)


In [ ]:
# @cr:code_system name='install_feature_engineering' id=94623932
# This cluster isn't a Databricks ML runtime, so the
# `databricks-feature-engineering` library isn't pre-installed.
# `_score_with_feature_store` (framework `batch_inference.py:495`)
# imports `FeatureEngineeringClient` / `FeatureLookup` from it, so the
# `refresh_predictions` cell would crash at import time without this.
# Runs ONCE per kernel session — `%pip install` auto-restarts the
# interpreter on DBR 13+; the explicit restartPython() is a
# belt-and-suspenders no-op on those runtimes and a hard requirement
# on older ones. After restart, re-run from the next cell down.
%pip install databricks-feature-engineering --quiet
dbutils.library.restartPython()  # noqa: F821 — Databricks notebook builtin


[//]: # (cr:doc name='chapter_c04_batch_inference' id=01b14776)
# Chapter c04: Batch Inference (Causal Track)

Refreshes the `predictions` Delta table by scoring the current feature-store snapshot against the registered `@production` model. Independent from c02 (archetypes) and c05 (snapshot + dashboard) so operators can re-run scoring without recomputing archetypes.

`BATCH_INFERENCE_MODE='auto'` (default) scores only when `predictions` is missing or older than `PREDICTIONS_STALE_AFTER_HOURS`. Use `'always'` to force a fresh scoring run or `'never'` to skip (useful when inspecting the model without writing).


In [ ]:
# @cr:code name='init_progress' id=49f15248
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c04_batch_inference.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c04_configuration' id=e124bf24)
## Configuration

The cell below is the only place you should need to edit. Every value here is read by the batch-inference cell — nothing is hardcoded inside the algorithmic cells.

- **`BATCH_INFERENCE_MODE`** — `"auto"` (default): run only if `predictions` is missing or stale; `"always"`: force a scoring run regardless of freshness; `"never"`: skip entirely.
- **`PREDICTIONS_STALE_AFTER_HOURS`** — in `"auto"` mode, re-score if the latest `inference_point_in_time` in `predictions` is older than this many hours.
- **`SCORING_THRESHOLD`** — probability cutoff that marks a row as a predicted churner. Must match the threshold used in `decision_policy`.
- **`RISK_TIER_HIGH` / `RISK_TIER_MEDIUM`** — risk-tier cutoffs written onto each scored row; c05's snapshot writer reads these back when `SNAPSHOT_RISK_TIER_*` are `None`.


In [ ]:
# @cr:config name='configuration' id=99787c69
BATCH_INFERENCE_MODE = "auto"           # "auto" | "always" | "never"
PREDICTIONS_STALE_AFTER_HOURS = 24

SCORING_THRESHOLD = 0.5
RISK_TIER_HIGH = 0.6
RISK_TIER_MEDIUM = 0.3

# === Optional engagement overrides — leave None for auto-detection ====
# RunNamespace.resolve() honours these first, then falls back to
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime).
# Pin them when several runs share an experiments dir and auto-detection
# picked the wrong one.
ENGAGEMENT_RUN_ID = None
ENGAGEMENT_EXPERIMENTS_DIR = None
MODEL_URI_OVERRIDE = None  # if None, looked up from MLflow @production alias
# Optional engagement catalog/schema pins. Default None → resolve from the
# active run's ScoringConfig (``CR_CATALOG`` / ``CR_SCHEMA`` env vars, or
# the persisted ``.churnkit_config.json``). Set these to bypass discovery.
ENGAGEMENT_CATALOG = None
ENGAGEMENT_SCHEMA = None

# Entities ever at the positive outcome (e.g. any landing row with
# ``churned = 1``) are unconditionally dropped from the scoring cohort by
# ``_resolve_already_positive_exclusion`` + a ``left_anti`` join against
# ``landing_<target>`` in the refresh cell below. The exclusion is
# entity-level — a row-level ``churned = 0`` predicate is wrong by
# construction against a snapshot-panel landing table, because an entity
# with both 0-rows and 1-rows survives ``distinct(entity_id)`` and slips
# through. Resolution is driven by ``ProjectContext.target_column`` and
# the target dataset's ``entity_column``; there is no opt-out knob,
# because the trained model has nothing useful to say about an already-
# positive row and the CSM-facing dashboard would otherwise surface
# accounts the business can no longer recover. To stop scoring entirely
# set ``BATCH_INFERENCE_MODE = "never"`` above.

[//]: # (cr:doc name='c04_batch_inference_setup' id=e4fb8e8b)
## 4.0 Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=3c7ff9a0
from customer_retention.core.compat.detection import get_spark_session, is_databricks

try:
    from customer_retention.core.config import get_playbooks_dir
except ImportError:
    # Stale cluster wheels (pre-`get_playbooks_dir` package-level export)
    # ship the symbol only on the experiments submodule. Fall back so
    # the cell runs without forcing a wheel reinstall before scoring.
    from customer_retention.core.config.experiments import get_playbooks_dir
from customer_retention.stages.scoring import resolve_scoring_context

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

# Auto-detect the active run / model via RunNamespace's file-tracked
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime). On
# Databricks, the model URI is resolved from MLflow's @production alias
# for the registered model named in training_metadata.json. Operator
# overrides from the configuration cell above (ENGAGEMENT_RUN_ID /
# ENGAGEMENT_EXPERIMENTS_DIR / MODEL_URI_OVERRIDE) short-circuit each
# resolution tier when set; default None → auto-detect.
_resolved = resolve_scoring_context(
    run_id=globals().get("ENGAGEMENT_RUN_ID"),
    experiments_dir=globals().get("ENGAGEMENT_EXPERIMENTS_DIR"),
    model_uri=globals().get("MODEL_URI_OVERRIDE"),
)
scoring_config = _resolved.scoring_config
_namespace = _resolved.namespace
_ns_source = _resolved.source
CATALOG = scoring_config.catalog if is_databricks() else "local"
SCHEMA = scoring_config.schema if is_databricks() else "local"
MODEL_NAME = _resolved.model_name
MODEL_VERSION = _resolved.model_version
MODEL_URI = _resolved.model_uri

COMPOSITE_NAME = scoring_config.composite_name
GOLD_FEATURES_FQN = (
    f"{CATALOG}.{SCHEMA}.gold_features_{COMPOSITE_NAME}"
    if COMPOSITE_NAME
    else f"{CATALOG}.{SCHEMA}.gold_features"
)

ARCHETYPE_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.archetype_catalog"
ELIGIBILITY_POLICY_FQN = f"{CATALOG}.{SCHEMA}.eligibility_policy"
DECISION_POLICY_FQN = f"{CATALOG}.{SCHEMA}.decision_policy"
ELIGIBILITY_SNAPSHOT_FQN = f"{CATALOG}.{SCHEMA}.eligibility_snapshot"
PREDICTIONS_FQN = f"{CATALOG}.{SCHEMA}.predictions"
TOP_SHAP_DRIVERS_FQN = f"{CATALOG}.{SCHEMA}.top_shap_drivers"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Active run namespace:   {_namespace.run_id if _namespace else '(none)'}")
print(f"Run source:             {_ns_source}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")
print(f"Composite name:         {COMPOSITE_NAME or '(unset)'}")
print(f"Gold features table:    {GOLD_FEATURES_FQN}")
print(f"Model URI:              {MODEL_URI or '(local)'}")
print(f"Model version:          {MODEL_VERSION}")


In [ ]:
# @cr:probe name='scoring_path_smoke' id=fc93bd73
# Two-phase probe.
#
# Phase 1 (static): import surface + MLflow model metadata. Predicts which
# code path _score_with_feature_store will choose without running anything.
#
# Phase 2 (dry-run): if Phase 1 says the plain-spark fallback will fire,
# exercise that fallback end-to-end on a 5-entity sample BEFORE the real
# refresh_predictions burns through 78k+ entities. Catches the three real
# failure modes downstream of the framework's try/except:
#   (a) customer_features table missing or wrong schema  -> create_training_set fails
#   (b) pyfunc-loaded Spark ML model expects vector input -> predict fails
#   (c) feature_cols count or order mismatches the model  -> predict fails
# All three are silent in v3.X without this probe; they only surface as
# multi-minute scoring failures.
print("=" * 70)
print("[v3 §X] c04 scoring-path smoke probe (Phase 1: static)")
print("=" * 70)

# ---- Phase 1 ----
try:
    from databricks.feature_engineering import (  # noqa: F401 — importability probe
        FeatureEngineeringClient,
        FeatureLookup,
    )
    _have_fe = True
    print("[1/4] databricks.feature_engineering         : importable")
except ImportError as _exc:
    _have_fe = False
    print(f"[1/4] databricks.feature_engineering         : MISSING ({_exc})")

import mlflow

_flavors, _loader, _fe_wrapped = [], None, False
try:
    _info = mlflow.models.get_model_info(MODEL_URI)
    _flavors = sorted(_info.flavors.keys())
    _loader = _info.flavors.get("python_function", {}).get("loader_module", "")
    _flavors_blob = str(_info.flavors).lower()
    _fe_wrapped = any(
        marker in _flavors_blob
        for marker in ("feature_engineering", "feature_spec",
                       "databricks_feature_store", "training_set")
    )
    print(f"[2/4] model flavors                          : {_flavors}")
    print(f"[2/4] python_function.loader_module          : {_loader or '(unset)'}")
    print(f"[2/4] FE-training-set wrapper detected       : {_fe_wrapped}")
except Exception as _exc:
    print(f"[2/4] could not read model metadata          : {_exc}")

print("[3/4] framework code path: try fe.score_batch -> fallback to")
print("       fe.create_training_set + mlflow.pyfunc.load_model + .predict")

_dry_run_needed = _have_fe and not _fe_wrapped
if not _have_fe:
    print("[4/4] FAIL Phase 1 — FE library missing; cell 0 install must succeed first.")
elif _fe_wrapped:
    print("[4/4] PASS Phase 1 — FE-wrapped model + library available; fe.score_batch path expected.")
    print("       Skipping Phase 2 (no fallback to exercise).")
else:
    print("[4/4] PASS Phase 1 — plain-spark model + library available; fallback path will run.")
    print("       Running Phase 2 dry-run on a 5-entity sample to validate it actually works.")

# ---- Phase 2 ----
if _dry_run_needed:
    print()
    print("=" * 70)
    print("[v3 §X] Phase 2: dry-run fallback on 5-entity sample")
    print("=" * 70)

    from pyspark.sql import functions as _F  # noqa: N812 — standard PySpark alias

    # Resolve the same tables run_batch_inference will use
    _customer_table_fqn = GOLD_FEATURES_FQN
    _timestamp_col = scoring_config.timestamp_column

    # 2a: customer_features existence + schema
    _ft_fqn = f"{CATALOG}.{SCHEMA}.customer_features"
    try:
        _ft_exists = spark.catalog.tableExists(_ft_fqn)
        print(f"[2a]  feature_table.exists  : {_ft_fqn} -> {_ft_exists}")
        if not _ft_exists:
            # The framework's run_batch_inference may default feature_table=customer_table,
            # in which case the join is against gold_features directly. Re-check that fallback.
            _ft_fqn = _customer_table_fqn
            _ft_exists = spark.catalog.tableExists(_ft_fqn)
            print(f"[2a]  feature_table fallback to customer_table -> {_ft_fqn} exists={_ft_exists}")
        if _ft_exists:
            _ft = spark.table(_ft_fqn)
            print(f"[2a]  feature_table columns: {len(_ft.columns)} cols (first 8): {_ft.columns[:8]}")
            assert "entity_id" in _ft.columns, f"entity_id missing from {_ft_fqn}"
            print("[2a]  entity_id present   : True")
            if _timestamp_col in _ft.columns:
                print(f"[2a]  {_timestamp_col} present   : True")
            else:
                print(f"[2a]  WARN {_timestamp_col} NOT in feature_table -> FeatureLookup may fail at create_training_set")
        else:
            raise RuntimeError("neither customer_features nor gold_features_<CN> resolved")
    except Exception as _exc:
        print(f"[2a]  FAIL feature_table preflight: {_exc}")
        raise

    # 2b: pyfunc load + signature inspection
    print()
    try:
        # Use mlflow.spark.load_model with dfs_tmpdir to satisfy the
        # shared/serverless cluster's UC-volume requirement. (Bare
        # mlflow.pyfunc.load_model doesn't accept dfs_tmpdir and fails
        # with "UC volume path must be provided" on those runtimes;
        # that's exactly what §Y's bypass cell works around at score time.)
        import mlflow.spark as _v3x_mspark
        _v3x_dfs_tmpdir = f"/Volumes/{ENGAGEMENT_CATALOG}/{ENGAGEMENT_SCHEMA}/_mlflow_tmp"
        _model_pipeline = _v3x_mspark.load_model(MODEL_URI, dfs_tmpdir=_v3x_dfs_tmpdir)
        print(f"[2b]  mlflow.spark.load_model OK (dfs_tmpdir={_v3x_dfs_tmpdir})")
        print(f"[2b]  loaded type: {type(_model_pipeline).__name__}")
        _stages = getattr(_model_pipeline, "stages", None)
        if _stages:
            print(f"[2b]  pipeline stages ({len(_stages)}): {[type(s).__name__ for s in _stages]}")
    except Exception as _exc:
        print(f"[2b]  FAIL spark.load_model: {_exc}")
        raise

    # 2c: 5-entity dry-run join + predict
    print()
    try:
        # Spark-ML-native dry-run: latest-row-per-entity on 5 entities,
        # PipelineModel.transform — mirrors what §Y's bypass does at full scale.
        _sample = spark.table(_customer_table_fqn).select("entity_id").distinct().limit(5)
        _gold_5 = (
            spark.table(_ft_fqn)
                 .join(_sample, on="entity_id", how="inner")
        )
        from pyspark.sql.window import Window as _v3x_W
        _w = _v3x_W.partitionBy("entity_id").orderBy(_F.col(_timestamp_col).desc())
        _aligned = (
            _gold_5
            .withColumn("__v3x_rn", _F.row_number().over(_w))
            .filter(_F.col("__v3x_rn") == 1)
            .drop("__v3x_rn")
        )
        print(f"[2c]  aligned latest-per-entity: {_aligned.count()} rows, {len(_aligned.columns)} cols")
        _scored = _model_pipeline.transform(_aligned)
        _scored_n = _scored.count()
        print(f"[2c]  PipelineModel.transform OK: {_scored_n} rows scored")
        print(f"[2c]  output columns: {[c for c in _scored.columns if c in ('prediction','probability','rawPrediction')]}")
        print(f"[2c]  PASS Phase 2 — §Y bypass will succeed on the full {scoring_config.target_column} run.")
    except Exception as _exc:
        print(f"[2c]  FAIL Spark-ML dry-run: {type(_exc).__name__}: {_exc}")
        print()
        print("       §Y bypass would also fail with this error in refresh_predictions.")
        print("       Common causes:")
        print("         - PipelineModel needs columns absent from gold_features (training/serving skew)")
        print("         - cast / nullability mismatch between gold schema and the pipeline's expected input")
        raise
print("=" * 70)


In [ ]:
# @cr:user_code name='sparkml_scoring_bypass' id=b11edd97
# Monkey-patch _score_with_feature_store with a Spark-ML-native scorer
# that works on Databricks shared/serverless clusters. See markdown cell
# above for the why. This cell MUST run before refresh_predictions.
import mlflow.spark as _v3y_mspark
from pyspark.ml.functions import vector_to_array as _v3y_v2a
from pyspark.sql import functions as _v3y_F  # noqa: N812 — standard PySpark alias
from pyspark.sql.window import Window as _v3y_W

from customer_retention.stages.scoring import batch_inference as _v3y_bi


def _v3y_score_with_feature_store(
    spark,
    entity_df,
    feature_table,
    model_uri,
    timestamp_column="event_timestamp",
):
    """Spark-ML-native replacement for the framework's _score_with_feature_store.

    Contract (matches what run_batch_inference at lines 373-393 expects):
      - Input entity_df: Spark DataFrame with at least 'entity_id' (and a
        'timestamp_column' filled with the inference timestamp, ignored here
        because we use latest-row-per-entity semantics).
      - Returns: Spark DataFrame with 'entity_id', timestamp_column, and a
        'prediction' double column carrying P(label=1).

    Feature-source resolution: the framework defaults
    BatchInferenceConfig.feature_table to "customer_features" — a separate
    FE-managed table that doesn't exist in this engagement's setup. When
    that table is absent we fall back to GOLD_FEATURES_FQN (the customer
    table that actually carries the feature vectors). Mirrors the Phase 2a
    fallback in the §X smoke probe above.
    """
    _dfs_tmpdir = f"/Volumes/{ENGAGEMENT_CATALOG}/{ENGAGEMENT_SCHEMA}/_mlflow_tmp"
    print(f"[v3 §Y] loading model via mlflow.spark with dfs_tmpdir={_dfs_tmpdir}")
    _model = _v3y_mspark.load_model(model_uri, dfs_tmpdir=_dfs_tmpdir)
    print(f"[v3 §Y] model loaded: {type(_model).__name__}")

    # Resolve the actual feature source. feature_table may be the
    # framework default ("customer_features") which doesn't exist here;
    # in that case use the gold features table the model was trained on.
    _ft_use = feature_table if spark.catalog.tableExists(feature_table) else GOLD_FEATURES_FQN
    if _ft_use != feature_table:
        print(f"[v3 §Y] feature_table '{feature_table}' missing — falling back to {_ft_use}")
    else:
        print(f"[v3 §Y] using feature source: {_ft_use}")

    # Latest gold row per entity_id. Window partitionBy(entity_id) order
    # by feature_timestamp DESC, take rn=1. This matches the framework's
    # "score current customers at the latest snapshot" intent and avoids
    # the FE library's create_training_set helper entirely.
    _gold = spark.table(_ft_use)
    _entity_ids = entity_df.select("entity_id").distinct()
    _candidates = _gold.join(_entity_ids, on="entity_id", how="inner")
    _w = _v3y_W.partitionBy("entity_id").orderBy(_v3y_F.col(timestamp_column).desc())
    _aligned = (
        _candidates
        .withColumn("__v3y_rn", _v3y_F.row_number().over(_w))
        .filter(_v3y_F.col("__v3y_rn") == 1)
        .drop("__v3y_rn")
    )
    print(f"[v3 §Y] aligned {_aligned.count():,} latest-per-entity rows for scoring")

    # PipelineModel.transform runs the WHOLE training pipeline: vector
    # assembler, scaler, classifier. Output adds 'rawPrediction',
    # 'probability' (Vector[2] = [P(0), P(1)]), and 'prediction' (the
    # predicted class label). We surface P(label=1) under the alias
    # 'prediction' so the framework's downstream code (which treats
    # the 'prediction' column as a probability) works unchanged.
    _scored = _model.transform(_aligned)
    return (
        _scored
        .withColumn("__v3y_prob_arr", _v3y_v2a(_v3y_F.col("probability")))
        .withColumn("prediction", _v3y_F.col("__v3y_prob_arr").getItem(1).cast("double"))
        .drop("__v3y_prob_arr", "probability", "rawPrediction")
        .select("entity_id", timestamp_column, "prediction")
    )


# Rebind on the framework module so run_batch_inference picks it up.
_v3y_bi._score_with_feature_store = _v3y_score_with_feature_store
print("[v3 §Y] _score_with_feature_store rebound to Spark-ML-native scorer")


[//]: # (cr:doc name='c04_refresh_section' id=4a814e0b)
## 4.1 Refresh Predictions


In [ ]:
# @cr:code name='refresh_predictions' id=d5f90d2d
from datetime import datetime, timezone

from customer_retention.stages.scoring.batch_inference import (
    BatchInferenceConfig,
    run_batch_inference,
)


def _resolve_scope_filter():
    """Return ``(filter_expr, landing_table_fqn, target_name)`` for the target
    dataset's NB00 ``ProjectContext.sample_filters`` entry. The filter narrows
    the scoring population to the entity subset in force during exploration /
    training. ``landing_table_fqn`` is the source table the framework routes
    the filter through — the customer table (gold) has categoricals one-hot
    encoded, so a filter referencing a raw landing column like
    ``REVENUE_MARKET_SEGMENT`` only resolves against ``landing_<target>``.
    Returns ``(None, None, None)`` when project_context is absent / empty."""
    try:
        from customer_retention.analysis.auto_explorer.project_context import ProjectContext
        from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace
    except ImportError:
        return None, None, None
    _ns = RunNamespace.from_env_or_latest()
    if _ns is None:
        return None, None, None
    _path = _ns.project_context_path
    if not _path.exists():
        return None, None, None
    _ctx = ProjectContext.load(_path)
    _filters = getattr(_ctx, "sample_filters", None) or {}
    if not _filters:
        return None, None, None
    _target_name = next(
        (name for name, ds in _ctx.datasets.items()
         if getattr(ds, "role", None) == "target"),
        None,
    )
    if _target_name is None and len(_ctx.datasets) == 1:
        _target_name = next(iter(_ctx.datasets))
    if _target_name is None:
        return None, None, None
    _filter_expr = _filters.get(_target_name)
    if not _filter_expr:
        return None, None, _target_name
    _landing_fqn = f"{CATALOG}.{SCHEMA}.landing_{_target_name}"
    return _filter_expr, _landing_fqn, _target_name


def _resolve_already_positive_exclusion():
    """Return ``(target_column, landing_fqn, raw_entity_key)`` so scoring can
    drop ANY entity whose target label has EVER been 1 in landing — applied
    by the framework as a ``left_anti`` join against
    ``landing_<target>.filter(<target_col> = 1)``.

    Entity-level by construction: a row-level ``<target_col> = 0`` clause
    cannot work against a snapshot-panel landing table, because an entity
    with both ``<target_col>=0`` and ``<target_col>=1`` rows would still
    survive the row-level filter (the 0-rows pass; ``distinct(entity_id)``
    re-admits the entity). Anti-join on the "ever positive" entity set is
    the only shape that matches the intent — drop the entity wholesale if
    landing has ANY row at the positive outcome.

    Always-on at scoring time: the trained model has nothing useful to say
    about an already-positive row, and surfacing such rows in the
    CSM-facing dashboard wastes headcount on accounts the business can no
    longer recover.

    Returns ``(None, None, None)`` when project_context is absent or
    cannot resolve a target column / raw entity key — the cell then falls
    back to the unmodified scope filter and logs the omission so the
    operator can see it didn't apply."""
    try:
        from customer_retention.analysis.auto_explorer.project_context import ProjectContext
        from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace
    except ImportError:
        return None, None, None
    _ns = RunNamespace.from_env_or_latest()
    if _ns is None or not _ns.project_context_path.exists():
        return None, None, None
    _ctx = ProjectContext.load(_ns.project_context_path)
    _target_col = getattr(_ctx, "target_column", None)
    if not _target_col:
        return None, None, None
    _target_name = (
        getattr(_ctx, "target_dataset", None)
        or next(
            (n for n, d in _ctx.datasets.items() if getattr(d, "role", None) == "target"),
            None,
        )
    )
    if _target_name is None and len(_ctx.datasets) == 1:
        _target_name = next(iter(_ctx.datasets))
    if _target_name is None:
        return None, None, None
    _target_ds = _ctx.datasets.get(_target_name)
    _raw_key = getattr(_target_ds, "entity_column", None) if _target_ds is not None else None
    if not _raw_key:
        return None, None, None
    _landing_fqn = f"{CATALOG}.{SCHEMA}.landing_{_target_name}"
    return _target_col, _landing_fqn, _raw_key


_scope_filter, _filter_via_table, _target_dataset_name = _resolve_scope_filter()
_excl_target_col, _excl_landing_fqn, _excl_entity_key = _resolve_already_positive_exclusion()

batch_inference_result = None
_predictions_status = "UNKNOWN"

if spark is None:
    _predictions_status = "SKIPPED: no Spark session (Databricks-only cell)"
    print(_predictions_status)
elif BATCH_INFERENCE_MODE == "never":
    _predictions_status = "SKIPPED: BATCH_INFERENCE_MODE='never'"
    print(_predictions_status)
else:
    _should_run = True
    _stale_reason = "always" if BATCH_INFERENCE_MODE == "always" else None
    if BATCH_INFERENCE_MODE == "auto":
        if not spark.catalog.tableExists(PREDICTIONS_FQN):
            _stale_reason = f"{PREDICTIONS_FQN} does not exist"
        else:
            _latest_ts_row = spark.sql(
                f"SELECT max(inference_point_in_time) AS ts FROM {PREDICTIONS_FQN}"
            ).head()
            _latest_ts = _latest_ts_row["ts"] if _latest_ts_row is not None else None
            if _latest_ts is None:
                _stale_reason = f"{PREDICTIONS_FQN} is empty"
            else:
                if _latest_ts.tzinfo is None:
                    _latest_ts = _latest_ts.replace(tzinfo=timezone.utc)
                _age_hours = (datetime.now(timezone.utc) - _latest_ts).total_seconds() / 3600.0
                if _age_hours > PREDICTIONS_STALE_AFTER_HOURS:
                    _stale_reason = f"latest inference_point_in_time is {_age_hours:.1f}h old (> {PREDICTIONS_STALE_AFTER_HOURS}h)"
                else:
                    _should_run = False
                    _predictions_status = (
                        f"FRESH: latest inference_point_in_time is {_age_hours:.1f}h old "
                        f"(<= {PREDICTIONS_STALE_AFTER_HOURS}h) — skipping"
                    )
                    print(_predictions_status)
    if _should_run:
        # When the scope filter is set, route it through `landing_<target>`
        # so raw categorical columns (one-hot encoded in gold) still resolve.
        # When `landing_<target>` does not exist (rare — only if landing was
        # never written), fall back to direct gold filter and let Spark
        # raise UNRESOLVED_COLUMN with the suggested one-hot alternatives.
        _resolved_filter_via_table = None
        if _scope_filter and _filter_via_table and spark.catalog.tableExists(_filter_via_table):
            _resolved_filter_via_table = _filter_via_table

        print(f"Running batch inference ({_stale_reason})")
        if _scope_filter:
            print(f"Scope filter (from NB00 project_context): {_scope_filter}")
            if _resolved_filter_via_table:
                print(f"  Routed via: {_resolved_filter_via_table} (entity_id inner-join with gold)")
            else:
                print(f"  Routed via: gold directly (landing_{_target_dataset_name or '?'} not found)")
        else:
            print("Scope filter: (none — scoring full entity population)")
        if _excl_target_col and _excl_landing_fqn and _excl_entity_key:
            print(
                f"Already-positive exclusion (entity-level anti-join): "
                f"DROP entity where any row of {_excl_landing_fqn} has "
                f"{_excl_target_col} = 1 (key={_excl_entity_key})"
            )
        else:
            print(
                "Already-positive exclusion: (none — project_context.target_column "
                "or target dataset entity_column not resolvable; entities already "
                "at the positive outcome may be scored)"
            )
        config = BatchInferenceConfig(
            catalog=CATALOG,
            schema=SCHEMA,
            model_uri=MODEL_URI,
            customer_table=GOLD_FEATURES_FQN,
            threshold=SCORING_THRESHOLD,
            risk_tier_high=RISK_TIER_HIGH,
            risk_tier_medium=RISK_TIER_MEDIUM,
            inference_timestamp=datetime.now(timezone.utc),
            filter_expression=_scope_filter,
            filter_via_table=_resolved_filter_via_table,
            exclude_already_positive_target_column=_excl_target_col,
            exclude_already_positive_via_table=_excl_landing_fqn,
            exclude_already_positive_entity_key=_excl_entity_key,
        )
        batch_inference_result = run_batch_inference(config)
        _predictions_status = batch_inference_result.summary()
        print(batch_inference_result.long_summary())

[//]: # (cr:doc name='c04_summary_section' id=0910a8d7)
## 4.2 Print Run Summary


In [ ]:
# @cr:code name='print_run_summary' id=429afe3b
if batch_inference_result is None:
    print(f"Predictions status: {_predictions_status}")
else:
    print(f"Inference id: {batch_inference_result.inference_id}")
    print(f"Inference timestamp: {batch_inference_result.inference_timestamp}")
    print(f"Scored: {batch_inference_result.total_scored:,}")
    print(f"Predicted churners: {batch_inference_result.predicted_churners:,}")
    print(f"Mean probability: {batch_inference_result.avg_probability:.4f}")
    print(f"Target table: {batch_inference_result.target_table_fqn}")


[//]: # (cr:doc name='c04_leakage_probe_section' id=2793f2a6)
## 4.3 Already-positive leakage verification (independent probe)

Independent sanity check that runs after `refresh_predictions`: queries `landing_<target>`, `predictions`, and the latest `eligibility_snapshot` partition directly — **without** going through the framework's scope-filter resolver — and counts how many entities whose target label is already 1 (e.g. `churned = 1`) survived into the scored output and the dashboard read surface. Treat any non-zero leakage as a regression — the always-on exclusion in the refresh cell should have dropped them before scoring. Adds one cheap aggregation per Delta table and prints a small sample of leaked entity_ids when the count is non-zero.

In [ ]:
# @cr:probe name='already_positive_leakage_probe' id=fefc408b
"""Independent verification that the always-on exclusion actually fired.

Reads landing_<target>, predictions, and eligibility_snapshot via plain
Spark SQL — no framework filter resolver, no project_context-driven
routing — and reports how many entities whose target label is already 1
(e.g. churned = 1) survived into the scored output and the latest
snapshot the dashboard reads. The setup cell already loaded
project_context for run resolution, so this cell just looks up
target_dataset / target_column from it and runs three aggregations."""
from pyspark.sql import functions as F  # noqa: N812 — standard alias

from customer_retention.analysis.auto_explorer.project_context import ProjectContext
from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace

_probe_ns = (
    globals().get("_namespace")
    or globals().get("_ns")
    or (RunNamespace.from_env_or_latest() if spark is not None else None)
)
if _probe_ns is None or not _probe_ns.project_context_path.exists():
    print("[probe] no run namespace / project_context — skipping leakage probe")
else:
    _probe_ctx = ProjectContext.load(_probe_ns.project_context_path)
    _probe_target_col = getattr(_probe_ctx, "target_column", None)
    _probe_target_ds = next(
        (n for n, d in _probe_ctx.datasets.items() if getattr(d, "role", None) == "target"),
        None,
    ) or (next(iter(_probe_ctx.datasets)) if len(_probe_ctx.datasets) == 1 else None)

    if not (_probe_target_col and _probe_target_ds):
        print(f"[probe] target_column={_probe_target_col!r} target_dataset={_probe_target_ds!r} — cannot probe")
    else:
        _probe_landing = f"{CATALOG}.{SCHEMA}.landing_{_probe_target_ds}"
        _probe_pred = f"{CATALOG}.{SCHEMA}.predictions"
        _probe_snap = f"{CATALOG}.{SCHEMA}.eligibility_snapshot"
        print("=" * 70)
        print("[probe] already-positive leakage check")
        print(f"  target_column     : {_probe_target_col}")
        print(f"  landing           : {_probe_landing}")
        print(f"  predictions       : {_probe_pred}")
        print(f"  eligibility_snap  : {_probe_snap}")
        print("=" * 70)

        if not spark.catalog.tableExists(_probe_landing):
            print(f"[probe] {_probe_landing} does not exist; cannot run probe")
        else:
            _land_cols = [f.name for f in spark.table(_probe_landing).schema.fields]
            if _probe_target_col not in _land_cols:
                print(
                    f"[probe] WARNING: {_probe_target_col!r} NOT in {_probe_landing} schema "
                    f"(cols: {_land_cols[:8]}{'...' if len(_land_cols) > 8 else ''})"
                )
                print("        the always-on `target_column = 0` clause was a NO-OP at scoring time.")
                print("        likely cause: the target was derived in exploration via a `@cr:user_code`")
                print("        cell that NB10 didn't replay into production landing.")
            else:
                _ds_obj = _probe_ctx.datasets.get(_probe_target_ds)
                _raw_key = getattr(_ds_obj, "entity_column", None)
                _probe_entity_key = "entity_id" if "entity_id" in _land_cols else _raw_key
                if not _probe_entity_key or _probe_entity_key not in _land_cols:
                    print(
                        f"[probe] could not resolve entity key on {_probe_landing} "
                        f"(raw={_raw_key!r}, cols={_land_cols[:6]}...)"
                    )
                else:
                    # One row per entity: ever_positive=1 if ANY landing record
                    # for that entity has target=1. Conservative: if the entity
                    # was ever flagged churned, we count it as already-positive.
                    _land_entity = (
                        spark.table(_probe_landing)
                        .groupBy(F.col(_probe_entity_key).alias("entity_id"))
                        .agg(F.max(F.col(_probe_target_col).cast("int")).alias("ever_positive"))
                    )
                    _land_summary = _land_entity.agg(
                        F.count("*").alias("total"),
                        F.sum("ever_positive").alias("positive"),
                    ).head()
                    _land_total = int(_land_summary["total"] or 0)
                    _land_positive = int(_land_summary["positive"] or 0)
                    print("[probe] landing population (one row per entity, max(target)):")
                    print(f"  distinct entities         : {_land_total:,}")
                    print(f"  ever {_probe_target_col} = 1            : {_land_positive:,}")

                    if not spark.catalog.tableExists(_probe_pred):
                        print(f"\n[probe] {_probe_pred} does not exist; skipping predictions overlap")
                    else:
                        _pred_df = spark.table(_probe_pred).select("entity_id").distinct()
                        _pred_join = _pred_df.join(_land_entity, "entity_id", "inner")
                        _pred_row = _pred_join.agg(
                            F.count("*").alias("matched"),
                            F.sum("ever_positive").alias("positive_scored"),
                        ).head()
                        _matched = int(_pred_row["matched"] or 0)
                        _leaked = int(_pred_row["positive_scored"] or 0)
                        _verdict = "<--- LEAKED" if _leaked > 0 else "OK"
                        print("\n[probe] predictions overlap with landing:")
                        print(f"  scored entities matched      : {_matched:,}")
                        print(f"  with ever {_probe_target_col} = 1          : {_leaked:,}  {_verdict}")

                    if not spark.catalog.tableExists(_probe_snap):
                        print(f"\n[probe] {_probe_snap} does not exist; skipping snapshot overlap")
                    else:
                        _snap_df = spark.table(_probe_snap)
                        _max_date = _snap_df.agg(F.max("as_of_date").alias("d")).head()["d"]
                        _snap_latest = (
                            _snap_df.filter(F.col("as_of_date") == F.lit(_max_date))
                            .select("entity_id").distinct()
                        )
                        _snap_join = _snap_latest.join(_land_entity, "entity_id", "inner")
                        _snap_row = _snap_join.agg(
                            F.count("*").alias("matched"),
                            F.sum("ever_positive").alias("positive_in_snapshot"),
                        ).head()
                        _smatched = int(_snap_row["matched"] or 0)
                        _sleaked = int(_snap_row["positive_in_snapshot"] or 0)
                        _sverdict = "<--- LEAKED to dashboard" if _sleaked > 0 else "OK"
                        print(f"\n[probe] eligibility_snapshot overlap (as_of_date={_max_date}):")
                        print(f"  distinct entities in snapshot   : {_smatched:,}")
                        print(f"  with ever {_probe_target_col} = 1             : {_sleaked:,}  {_sverdict}")
                        if _sleaked > 0:
                            _leaked_sample = (
                                _snap_latest
                                .join(_land_entity.filter(F.col("ever_positive") == 1),
                                      "entity_id", "inner")
                                .select("entity_id").limit(20).collect()
                            )
                            print("  sample of leaked entity_ids (up to 20):")
                            for _r in _leaked_sample:
                                print(f"    {_r['entity_id']}")
        print("=" * 70)

In [ ]:
# @cr:code name='release_stage_memory' id=2a6327db
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
